In [21]:
import numpy as np
from sklearn import svm
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

# 1. Prepare your data
# X should be your (120, 7) array of binary symptoms
# y should be your (120,) array of labels (0 or 1)
## replace Positive as 1 and Negative as 0
df = pd.read_csv('dataset_120.csv')
df = df.drop(columns=['patient_id'])
mapping_dict = {'Positive': 1, 'Negative': 0}
df['label'] = df['label'].map(mapping_dict)
df.to_csv('updated_dataset_120.csv', index=False)

## split train test dataset in stratified way. Train and test dataset have equal proportion of positives and negatives
features = df.iloc[:, 2: 9]
labels = df.iloc[:, 1]

X = np.array(features) 
y = np.array(labels)
correlation_matrix = features.corr()
print(correlation_matrix)


# 2. Split the data (using stratify to keep class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# print(X_train, y_train)
# 3. Hyperparameter Tuning with GridSearchCV
# For small datasets, 'C' and 'gamma' are critical to prevent overfitting
param_grid = {
    'C': [0.1, 1, 10, 100],          # Regularization: Low C = wider margin (prevents overfitting)
    'gamma': [1, 0.1, 0.01, 0.001], # Influence of single training points
    'kernel': ['rbf', 'linear']     # RBF is great for non-linear clinical patterns
}

grid = GridSearchCV(svm.SVC(), param_grid, refit=True, verbose=0, cv=5)
grid.fit(X_train, y_train)

# 4. Evaluate the model
best_svc = grid.best_estimator_
y_pred = best_svc.predict(X_test)

print(f"Best Parameters: {grid.best_params_}")
print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.2f}%")
print(classification_report(y_test, y_pred))


                    Hoarseness  Rhinorrhea  sorethroat  Congestion  \
Hoarseness            1.000000    0.374876    0.123602    0.303777   
Rhinorrhea            0.374876    1.000000   -0.115636    0.484581   
sorethroat            0.123602   -0.115636    1.000000    0.028475   
Congestion            0.303777    0.484581    0.028475    1.000000   
Knownrecentcontact    0.143228    0.179177   -0.034627    0.138118   
Headache              0.216007    0.062696    0.111072    0.141918   
Fever                -0.061563    0.099471   -0.028475    0.091219   

                    Knownrecentcontact  Headache     Fever  
Hoarseness                    0.143228  0.216007 -0.061563  
Rhinorrhea                    0.179177  0.062696  0.099471  
sorethroat                   -0.034627  0.111072 -0.028475  
Congestion                    0.138118  0.141918  0.091219  
Knownrecentcontact            1.000000  0.011317  0.033191  
Headache                      0.011317  1.000000  0.229772  
Fever       

In [15]:
import torch
import pandas as pd
import numpy as np
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image
import cv2
import os

crop_size = 160
resize_img_dim = (224, 224)

def white_balance(img):

    img = np.array(img).astype(float)

    ## average value of each channel
    avg_red = np.mean(img[:, :, 0])
    avg_green = np.mean(img[:, :, 1])
    avg_blue = np.mean(img[:, :, 2])
    
    # gray value calculation
    avg_gray = (avg_red + avg_green + avg_blue) / 3
    
    # Scaling each channel
    img[:, :, 0] *= (avg_gray / avg_red)
    img[:, :, 1] *= (avg_gray / avg_green)
    img[:, :, 2] *= (avg_gray / avg_blue)
    
    # Clip values to [0, 255]
    img = np.clip(img, 0, 255).astype(np.uint8)     #the factor can cause values to go beyond 255, hence bounding it.
    return Image.fromarray(img)

def apply_clahe(img):
    img_np = np.array(img)
    lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l = clahe.apply(l)
    lab = cv2.merge((l,a,b))
    return Image.fromarray(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB))

class StrepDataset(Dataset):
    def __init__(self, csv, img_dir, transform = None):
        self.data = pd.read_csv(csv)
        self.img_dir = img_dir
        self.transform = transform
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        img_name = os.path.join(self.img_dir, self.data.iloc[idx, 0])
        image = Image.open(img_name).convert("RGB")

        if self.transform:
            image = self.transform(image)
        
        features = torch.tensor(self.data.iloc[idx, 2:9])

        label = torch.tensor(self.data.iloc[idx, 1])

        return image, features, label
    
transforms_train = transforms.Compose([
    transforms.Lambda(lambda x: white_balance(x)),
    # transforms.Lambda(lambda x: apply_clahe(x)),
    
    transforms.Resize(resize_img_dim),                        ## based on ResNet-18, the standard input image dim is (224, 224)
    transforms.CenterCrop(crop_size),                    ## Crop to focus only on the tonsil area, disregard tongue / teeth

    transforms.ToTensor(),
    # Use standard ResNet stats unless your custom_mean is based on 10,000+ images
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

indices = range(len(df))
train_idx, test_idx = train_test_split(
    indices, 
    test_size=0.3, 
    stratify=labels, 
    random_state=42
)

train_dataset = StrepDataset(csv='updated_dataset_120.csv', img_dir='data', transform=transforms_train)
train_subset = Subset(train_dataset, indices)
train_loader = DataLoader(train_subset, batch_size=16, shuffle=True)

In [16]:


# 1. Load your frozen ResNet (ImageNet or RadImageNet)
resnet = models.resnet18(weights='DEFAULT')
resnet.eval()
# Remove the final classification layer to get the 512 features
feature_extractor = torch.nn.Sequential(*list(resnet.children())[:-1])

def get_image_features(dataloader):
    all_features = []
    all_labels = []
    all_symptoms = []
    
    with torch.no_grad():
        for imgs, symp, lbls in train_loader:
            # Extract 512 numbers from the image
            feats = feature_extractor(imgs) 
            feats = torch.flatten(feats, 1).numpy()
            
            all_features.append(feats)
            all_symptoms.append(symp.numpy())
            all_labels.append(lbls.numpy())
            
    # Combine into a single matrix
    X_img = np.vstack(all_features)    # (120, 512)
    X_symp = np.vstack(all_symptoms)   # (120, 7)
    y = np.concatenate(all_labels)      # (120,)
    
    # Merge image features + 7 symptoms
    X_combined = np.hstack((X_img, X_symp)) # (120, 519)
    return X_combined, y

# Run this once on your full dataset
X, y = get_image_features(train_loader) 


C:\Users\avk1028\AppData\Local\Temp\ipykernel_17436\969638171.py:59: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  features = torch.tensor(self.data.iloc[idx, 2:9])


In [17]:
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler

# 1. Define the Pipeline
# We scale the data, reduce dimensions to 20, then classify
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=20)), # Reduces 519 features -> 20 patterns
    ('xgb', XGBClassifier(
        n_estimators=100, 
        max_depth=3,        # Shallow trees to prevent overfitting
        learning_rate=0.05, 
        subsample=0.8,      # Randomly use 80% of data for each tree
        use_label_encoder=False,
        eval_metric='logloss'
    ))
])

# 2. Use K-Fold Cross Validation (Crucial for 120 samples!)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y, cv=skf, scoring='accuracy')

print(f"Mean Validation Accuracy: {scores.mean()*100:.2f}%")
print(f"Standard Deviation: {scores.std()*100:.2f}%")


c:\Users\avk1028\AppData\Local\anaconda3\envs\pytorch_env\lib\site-packages\xgboost\training.py:200: UserWarning: [13:26:42] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\avk1028\AppData\Local\anaconda3\envs\pytorch_env\lib\site-packages\xgboost\training.py:200: UserWarning: [13:26:42] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\avk1028\AppData\Local\anaconda3\envs\pytorch_env\lib\site-packages\xgboost\training.py:200: UserWarning: [13:26:42] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\avk1028\AppData\Local\anaconda3\envs\pytorch_env\lib\site-packages\xgboost\training.py:200: UserWarning: [13:26:43] WARNI

Mean Validation Accuracy: 58.33%
Standard Deviation: 12.08%


c:\Users\avk1028\AppData\Local\anaconda3\envs\pytorch_env\lib\site-packages\xgboost\training.py:200: UserWarning: [13:26:43] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
